# 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, roc_curve, auc
from matplotlib.colors import ListedColormap
from sklearn.decomposition import PCA

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(len(np.unique(y_true)))
    plt.xticks(tick_marks, np.unique(y_true))
    plt.yticks(tick_marks, np.unique(y_true))
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    
    # Annotating the plot with the counts
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j],
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
    
    plt.show()

# 2. Load Dataset
We'll generate a synthetic dataset

In [ ]:
from sklearn.datasets import make_classification

# Create a synthetic dataset
X, y = make_classification(n_features=2, n_redundant=0, n_informative=1,
                           n_clusters_per_class=1, n_classes=2, class_sep=0.75, random_state=69)

#Split the data in Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)


# 3. Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

# Create meshgrid
h = .02
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))

In [ ]:


C_values = [0.001, 0.01, 0.1, 1, 10, 100,1000]
accuracies = []

# Determine the number of rows and columns for the subplots
n_cols = 2
n_rows = int(np.ceil(len(C_values) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 12))
axes = axes.flatten() # Flatten the axes array

for idx, C in enumerate(C_values):
    lr_model = LogisticRegression(C=C)
    lr_model.fit(X_train, y_train)
    y_pred = lr_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracies.append(accuracy)

    # Predict on meshgrid
    Z = lr_model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # Plot the decision boundary on the subplot
    axes[idx].contourf(xx, yy, Z, alpha=0.4, cmap=ListedColormap(('red', 'blue')))
    axes[idx].scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o', s=50, linewidth=1, cmap=ListedColormap(('red', 'blue')))
    axes[idx].set_title(f'C={C}')

# Remove any unused subplots
for idx in range(len(C_values), n_rows * n_cols):
    fig.delaxes(axes[idx])

plt.show()

plt.plot(C_values, accuracies,'bo--')
plt.xscale('log')
plt.xlabel('Regularization Strength (C)')
plt.ylabel('Accuracy')
plt.title('Logistic Regression: Accuracy vs C')
plt.show()

# Confusion Matrix
plot_confusion_matrix(y_test, y_pred, 'Logistic Regression Confusion Matrix')

y_prob = lr_model.predict_proba(X_test)[:, 1]

fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label='Logistic Regression (area = %0.2f)' % roc_auc)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()


# 4. Naïve Bayes

In [ ]:
from sklearn.naive_bayes import GaussianNB

# Create and fit the model
nb_model = GaussianNB()
nb_model.fit(X_train, y_train)

# Predict and evaluate
y_pred = nb_model.predict(X_test)
print('Naïve Bayes Accuracy:', accuracy_score(y_test, y_pred))

# Plot decision boundary
Z = nb_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.4, cmap=ListedColormap(('red', 'blue')))
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o', s=50, linewidth=1, cmap=ListedColormap(('red', 'blue')))
plt.title('Decision Boundary for Naïve Bayes')
plt.show()

# Confusion Matrix
plot_confusion_matrix(y_test, y_pred,"Naïve Bayes Confusion Matrix")

# ROC Curve
y_prob = nb_model.predict_proba(X_test)[:,1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label='Naïve Bayes (area = %0.2f)' % roc_auc)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()


# 5. K Nearest Neighbors

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

K_values = range(1, 15)
accuracies = []

for K in K_values:
    knn_model = KNeighborsClassifier(n_neighbors=K)
    knn_model.fit(X_train, y_train)
    y_pred = knn_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracies.append(accuracy)

# Plot accuracy vs K values
plt.plot(K_values, accuracies, 'bo--')
plt.xlabel('Number of Neighbors (K)')
plt.ylabel('Accuracy')
plt.title('K Nearest Neighbors: Accuracy vs K')
plt.show()

# Get the best K value
best_K = K_values[np.argmax(accuracies)]

# Train the best model
best_knn_model = KNeighborsClassifier(n_neighbors=best_K)
best_knn_model.fit(X_train, y_train)
y_pred = best_knn_model.predict(X_test)
y_prob = best_knn_model.predict_proba(X_test)[:,1]

# Decision boundary for best K
Z = best_knn_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)
plt.contourf(xx, yy, Z, alpha=0.4, cmap=ListedColormap(('red', 'blue')))
plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o', s=50, linewidth=1, cmap=ListedColormap(('red', 'blue')))
plt.title(f'Decision Boundary for K = {best_K}')
plt.show()

# Confusion Matrix
plot_confusion_matrix(y_test, y_pred, 'K Nearest Neighbors Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label='K Nearest Neighbors (area = %0.2f)' % roc_auc)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()


# 6. Decision Trees

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

# Create a synthetic dataset
X, y = make_classification(n_features=10, n_redundant=2, n_informative=4,
                           n_clusters_per_class=2, n_classes=2, class_sep=0.5, random_state=69)

# Split the data into Train and Test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

depth_values = range(1, 15)
accuracies = []

for depth in depth_values:
    dt_model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    dt_model.fit(X_train, y_train)
    y_pred = dt_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracies.append(accuracy)

# Plot accuracy vs depth values
plt.plot(depth_values, accuracies, 'bo--')
plt.xlabel('Max Depth')
plt.ylabel('Accuracy')
plt.title('Decision Trees: Accuracy vs Max Depth')
plt.show()

# Get the best depth value
best_depth = depth_values[np.argmax(accuracies)]

# Train the best model
best_dt_model = DecisionTreeClassifier(max_depth=best_depth, random_state=42)
best_dt_model.fit(X_train, y_train)
y_pred = best_dt_model.predict(X_test)
y_prob = best_dt_model.predict_proba(X_test)[:,1]

# Plot the decision tree
plt.figure(figsize=(15, 10))
plot_tree(best_dt_model, filled=True, rounded=True)
plt.show()

# Confusion Matrix
plot_confusion_matrix(y_test, y_pred, 'Decision Trees Confusion Matrix')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label='Decision Trees (area = %0.2f)' % roc_auc)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()

# 7. Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create a synthetic dataset
X, y = make_classification(n_features=2, n_redundant=0, n_informative=1,
                           n_clusters_per_class=1, n_classes=2, class_sep=0.75, random_state=69)

#Split the data in Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

estimators_values = [1,5, 7,10, 50, 100, 200, 500, 750, 1000]
accuracies = []

# Train Random Forest for different numbers of estimators
for estimators in estimators_values:
    rf_model = RandomForestClassifier(n_estimators=estimators, random_state=42)
    rf_model.fit(X_train, y_train)
    y_pred = rf_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracies.append(accuracy)

plt.plot(estimators_values, accuracies, 'bo--')
plt.xlabel('Number of Estimators')
plt.ylabel('Accuracy')
plt.title('Random Forest: Accuracy vs Number of Estimators')
plt.show()

# Plotting the first three trees from the Random Forest
for idx, tree in enumerate(rf_model.estimators_[:3]):
    plt.figure(figsize=(15, 10))
    plot_tree(tree, filled=True, rounded=True, class_names=['Class 0', 'Class 1'])
    plt.title(f'Tree {idx + 1}')
    plt.show()

# Confusion Matrix and ROC Curve for the last Random Forest model
plot_confusion_matrix(y_test, y_pred,'Random Forest Confusion Matrix')
y_prob = rf_model.predict_proba(X_test)[:, 1]
fpr, tpr, _ = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, label='Random Forest (area = %0.2f)' % roc_auc)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()


# 8. Support Vector Machine
Linear Kernel

In [ ]:
from sklearn.svm import SVC

# Create a synthetic dataset
X, y = make_classification(n_features=2, n_redundant=0, n_informative=1,
                           n_clusters_per_class=1, n_classes=2, class_sep=0.75, random_state=69)

#Split the data in Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

C_values = [0.001, 0.01, 0.1, 1, 10, 100, 500]
accuracies = []

# Create meshgrid for decision boundary
xx, yy = np.meshgrid(np.linspace(X[:, 0].min(), X[:, 0].max(), 100),
                     np.linspace(X[:, 1].min(), X[:, 1].max(), 100))

for C in C_values:
    svm_model = SVC(kernel='linear', C=C)
    svm_model.fit(X_train, y_train)
    y_pred = svm_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracies.append(accuracy)

    # Decision boundary
    Z = svm_model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.4, cmap=ListedColormap(('red', 'blue')))
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o', s=50, linewidth=1, cmap=ListedColormap(('red', 'blue')))
    plt.title(f'Decision Boundary for C = {C}')
    plt.show()

    # Get the best C value
best_C = C_values[np.argmax(accuracies)]

# Train the best model
best_svm_model = SVC(kernel='linear', C=best_C)
best_svm_model.fit(X_train, y_train)
y_pred = best_svm_model.predict(X_test)

plt.plot(C_values, accuracies,'bo--')
plt.xscale('log')
plt.xlabel('Regularization Parameter (C)')
plt.ylabel('Accuracy')
plt.title('Support Vector Machine: Accuracy vs C')
plt.show()

# Plot confusion matrix for the best model
plot_confusion_matrix(y_test, y_pred, title=f'Confusion Matrix for Best C = {best_C}')

Polynomial Kernel

In [ ]:
from sklearn.svm import SVC

# Create a synthetic dataset
X, y = make_classification(n_features=2, n_redundant=0, n_informative=1,
                           n_clusters_per_class=1, n_classes=2, class_sep=0.75, random_state=69)

#Split the data in Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

C_values = [0.001, 0.01, 0.1, 1, 10, 100,500]
accuracies = []

# Create meshgrid for decision boundary
xx, yy = np.meshgrid(np.linspace(X[:, 0].min(), X[:, 0].max(), 100),
                     np.linspace(X[:, 1].min(), X[:, 1].max(), 100))

for C in C_values:
    svm_model = SVC(kernel='poly', C=C)
    svm_model.fit(X_train, y_train)
    y_pred = svm_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracies.append(accuracy)

    # Decision boundary
    Z = svm_model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.4, cmap=ListedColormap(('red', 'blue')))
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o', s=50, linewidth=1, cmap=ListedColormap(('red', 'blue')))
    plt.title(f'Decision Boundary for C = {C}')
    plt.show()

    # Get the best C value
best_C = C_values[np.argmax(accuracies)]

# Train the best model
best_svm_model = SVC(kernel='linear', C=best_C)
best_svm_model.fit(X_train, y_train)
y_pred = best_svm_model.predict(X_test)

plt.plot(C_values, accuracies,'bo--')
plt.xscale('log')
plt.xlabel('Regularization Parameter (C)')
plt.ylabel('Accuracy')
plt.title('Support Vector Machine: Accuracy vs C')
plt.show()

# Plot confusion matrix for the best model
plot_confusion_matrix(y_test, y_pred, title=f'Confusion Matrix for Best C = {best_C}')

Radial Basis Function (RBF) Kernel

In [ ]:
from sklearn.svm import SVC

# Create a synthetic dataset
X, y = make_classification(n_features=2, n_redundant=0, n_informative=1,
                           n_clusters_per_class=1, n_classes=2, class_sep=0.75, random_state=69)

#Split the data in Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

C_values = [0.001, 0.01, 0.1, 1, 10, 100, 500]
accuracies = []

# Create meshgrid for decision boundary
xx, yy = np.meshgrid(np.linspace(X[:, 0].min(), X[:, 0].max(), 100),
                     np.linspace(X[:, 1].min(), X[:, 1].max(), 100))

for C in C_values:
    svm_model = SVC(kernel='rbf', C=C)
    svm_model.fit(X_train, y_train)
    y_pred = svm_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracies.append(accuracy)

    # Decision boundary
    Z = svm_model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.4, cmap=ListedColormap(('red', 'blue')))
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o', s=50, linewidth=1, cmap=ListedColormap(('red', 'blue')))
    plt.title(f'Decision Boundary for C = {C}')
    plt.show()

    # Get the best C value
best_C = C_values[np.argmax(accuracies)]

# Train the best model
best_svm_model = SVC(kernel='linear', C=best_C)
best_svm_model.fit(X_train, y_train)
y_pred = best_svm_model.predict(X_test)

plt.plot(C_values, accuracies,'bo--')
plt.xscale('log')
plt.xlabel('Regularization Parameter (C)')
plt.ylabel('Accuracy')
plt.title('Support Vector Machine: Accuracy vs C')
plt.show()

# Plot confusion matrix for the best model
plot_confusion_matrix(y_test, y_pred, title=f'Confusion Matrix for Best C = {best_C}')

Sigmoid Kernel

In [ ]:
from sklearn.svm import SVC

# Create a synthetic dataset
X, y = make_classification(n_features=2, n_redundant=0, n_informative=1,
                           n_clusters_per_class=1, n_classes=2, class_sep=0.75, random_state=69)

#Split the data in Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

C_values = [0.001, 0.01, 0.1, 1, 10, 100, 500]
accuracies = []

# Create meshgrid for decision boundary
xx, yy = np.meshgrid(np.linspace(X[:, 0].min(), X[:, 0].max(), 100),
                     np.linspace(X[:, 1].min(), X[:, 1].max(), 100))

for C in C_values:
    svm_model = SVC(kernel='sigmoid', C=C)
    svm_model.fit(X_train, y_train)
    y_pred = svm_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    accuracies.append(accuracy)

    # Decision boundary
    Z = svm_model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    plt.contourf(xx, yy, Z, alpha=0.4, cmap=ListedColormap(('red', 'blue')))
    plt.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', marker='o', s=50, linewidth=1, cmap=ListedColormap(('red', 'blue')))
    plt.title(f'Decision Boundary for C = {C}')
    plt.show()

    # Get the best C value
best_C = C_values[np.argmax(accuracies)]

# Train the best model
best_svm_model = SVC(kernel='linear', C=best_C)
best_svm_model.fit(X_train, y_train)
y_pred = best_svm_model.predict(X_test)

plt.plot(C_values, accuracies,'bo--')
plt.xscale('log')
plt.xlabel('Regularization Parameter (C)')
plt.ylabel('Accuracy')
plt.title('Support Vector Machine: Accuracy vs C')
plt.show()

# Plot confusion matrix for the best model
plot_confusion_matrix(y_test, y_pred, title=f'Confusion Matrix for Best C = {best_C}')